

| Secret name | Value |
|---|---|
| `DB_HOST` | e.g. `ep-xxxx.us-east-2.aws.neon.tech` (from Neon/Supabase) |
| `DB_PORT` | usually `5432` |
| `DB_NAME` | your database name |
| `DB_USER` | your database user |
| `DB_PASSWORD` | your database password |
| `JWT_SECRET` | any long random string (generate one in the next cell) |
| `SMTP_EMAIL` | your Gmail address |
| `SMTP_APP_PASSWORD` | 16-character Gmail **App Password** (not your real password) |
| `NGROK_AUTHTOKEN` | from https://dashboard.ngrok.com/get-started/your-authtoken |

**Note:** the FastAPI backend added below reuses `JWT_SECRET` — no additional secrets are needed for it.


In [1]:
!pip install -q streamlit psycopg2-binary PyJWT bcrypt \
    python-dotenv email-validator pyngrok \
    fastapi uvicorn python-multipart requests \
    langdetect ftfy emoji deep-translator vaderSentiment spacy pandas matplotlib \
    transformers accelerate torch stopwordsiso
!python -m spacy download xx_sent_ud_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 86.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 47.4 MB/s eta 0:00:00
✔ Downl

## New: Employee Wellness NLP Analysis

After login, the app now has an **"Run NLP Analysis"** button next to the file upload. It sends the CSV/TXT to a new `/analyze` endpoint on the FastAPI backend, which detects language (Telugu/Kannada-aware), cleans and tokenizes the text, translates it to English, lemmatizes, and runs VADER sentiment + keyword-based emotion detection. Results (including sentiment/emotion bar charts) render inline in Streamlit.

**No new secrets needed** — this reuses `JWT_SECRET` like the upload feature already did.

**Heads-up:** installing spaCy + downloading the `xx_sent_ud_sm` model adds a few minutes to Section 2's install cell the first time you run it in a fresh Colab runtime.


In [3]:
from google.colab import userdata

required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []
for key in required_secrets:
    try:
        values[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing Colab secrets: {missing}. "
        f"Add them via the key icon in the left sidebar, then re-run this cell."
    )

env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10
'''

with open(".env", "w") as f:
    f.write(env_content)

print("Wrote .env with", len(values), "secrets loaded.")

Wrote .env with 9 secrets loaded.


In [4]:
%%writefile db.py
import os, psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv
load_dotenv()

CFG = dict(host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT", "5432"),
           dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
           password=os.getenv("DB_PASSWORD"), sslmode="require")

@contextmanager
def cursor(commit=False):
    conn = psycopg2.connect(**CFG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        if commit: conn.commit()
    finally:
        cur.close(); conn.close()

def init_db():
    with cursor(commit=True) as cur:
        cur.execute("""CREATE TABLE IF NOT EXISTS users (
            id SERIAL PRIMARY KEY, username VARCHAR(50) UNIQUE, email VARCHAR(255) UNIQUE,
            password_hash VARCHAR(255), is_verified BOOLEAN DEFAULT FALSE)""")
        cur.execute("""CREATE TABLE IF NOT EXISTS otp_codes (
            id SERIAL PRIMARY KEY, email VARCHAR(255), code VARCHAR(6),
            purpose VARCHAR(20), expires_at TIMESTAMP, used BOOLEAN DEFAULT FALSE)""")

Writing db.py


In [5]:
%%writefile auth.py
import os, jwt, bcrypt, random, string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from db import cursor
load_dotenv()

SECRET = os.getenv("JWT_SECRET")

def hash_pw(pw): return bcrypt.hashpw(pw.encode(), bcrypt.gensalt()).decode()
def check_pw(pw, h): return bcrypt.checkpw(pw.encode(), h.encode())

def make_token(user):
    payload = {"id": user["id"], "username": user["username"], "email": user["email"],
               "exp": datetime.now(timezone.utc) + timedelta(hours=1)}
    return jwt.encode(payload, SECRET, algorithm="HS256")

def read_token(token):
    try: return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError: return None

def get_user(email):
    with cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email=%s", (email,))
        return cur.fetchone()

def username_taken(username):
    with cursor() as cur:
        cur.execute("SELECT 1 FROM users WHERE username=%s", (username,))
        return cur.fetchone() is not None

def create_user(username, email, pw):
    with cursor(commit=True) as cur:
        cur.execute("INSERT INTO users (username,email,password_hash) VALUES (%s,%s,%s)",
                    (username, email, hash_pw(pw)))

def verify_user(email):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified=TRUE WHERE email=%s", (email,))

def set_password(email, pw):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET password_hash=%s WHERE email=%s", (hash_pw(pw), email))

def new_otp():
    return "".join(random.choices(string.digits, k=6))

def save_otp(email, code, purpose):
    exp = datetime.now(timezone.utc) + timedelta(minutes=10)
    with cursor(commit=True) as cur:
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE email=%s AND purpose=%s", (email, purpose))
        cur.execute("INSERT INTO otp_codes (email,code,purpose,expires_at) VALUES (%s,%s,%s,%s)",
                    (email, code, purpose, exp))

def check_otp(email, code, purpose):
    with cursor(commit=True) as cur:
        cur.execute("""SELECT * FROM otp_codes WHERE email=%s AND purpose=%s AND used=FALSE
                       ORDER BY id DESC LIMIT 1""", (email, purpose))
        row = cur.fetchone()
        if not row or row["code"] != code:
            return False
        now = datetime.now(row["expires_at"].tzinfo) if row["expires_at"].tzinfo else datetime.now()
        if now > row["expires_at"]:
            return False
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE id=%s", (row["id"],))
        return True

Writing auth.py


In [6]:
%%writefile email_utils.py
import os, smtplib
from email.mime.text import MIMEText
from dotenv import load_dotenv
load_dotenv()

HOST, PORT = "smtp.gmail.com", 587
EMAIL = os.getenv("SMTP_EMAIL")
APP_PW = os.getenv("SMTP_APP_PASSWORD")

def send_otp(to_email, code, purpose):
    subject = "Your Verification Code" if purpose == "signup" else "Your Password Reset Code"
    msg = MIMEText(f"Your code is: {code}\nExpires in 10 minutes.")
    msg["From"], msg["To"], msg["Subject"] = EMAIL, to_email, subject
    try:
        with smtplib.SMTP(HOST, PORT, timeout=15) as s:
            s.starttls()
            s.login(EMAIL, APP_PW)
            s.sendmail(EMAIL, to_email, msg.as_string())
        return True, "sent"
    except Exception as e:
        return False, str(e)

Writing email_utils.py


In [7]:
%%writefile app.py
import os, re, time, sqlite3, base64, requests, streamlit as st
import plotly.graph_objects as go
from db import init_db
from auth import (make_token, read_token, get_user, username_taken, create_user,
                   verify_user, set_password, check_pw, new_otp, save_otp, check_otp)
from email_utils import send_otp

# ---------------------------------------------------------------------------
# Small self-contained store for things the existing db.py/auth.py schema
# doesn't cover yet (avatar persistence, in-app feedback). Kept in its own
# sqlite file so it never touches your existing user table/schema — once you
# add proper columns/tables for these in db.py, this can be swapped out.
# ---------------------------------------------------------------------------
EXTRA_DB = "app_extra.db"

def _extra_conn():
    conn = sqlite3.connect(EXTRA_DB)
    conn.execute("CREATE TABLE IF NOT EXISTS avatars (email TEXT PRIMARY KEY, avatar_b64 TEXT)")
    conn.execute("""CREATE TABLE IF NOT EXISTS feedback (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT, username TEXT, rating INTEGER, message TEXT, created_at REAL
    )""")
    return conn

def save_avatar(email, avatar_b64):
    conn = _extra_conn()
    conn.execute(
        "INSERT INTO avatars (email, avatar_b64) VALUES (?,?) "
        "ON CONFLICT(email) DO UPDATE SET avatar_b64=excluded.avatar_b64",
        (email, avatar_b64),
    )
    conn.commit(); conn.close()

def load_avatar(email):
    conn = _extra_conn()
    row = conn.execute("SELECT avatar_b64 FROM avatars WHERE email=?", (email,)).fetchone()
    conn.close()
    return row[0] if row else None

def save_feedback(email, username, rating, message):
    conn = _extra_conn()
    conn.execute(
        "INSERT INTO feedback (email, username, rating, message, created_at) VALUES (?,?,?,?,?)",
        (email, username, rating, message, time.time()),
    )
    conn.commit(); conn.close()

# ---------------------------------------------------------------------------
# Sidebar open/closed state
# ---------------------------------------------------------------------------
# Streamlit's sidebar collapse/expand state lives in the browser (a React
# aria-expanded flag), not in Python, and pure CSS overrides get silently
# re-stomped by Streamlit's own re-renders. The only way Streamlit actually
# picks up a *forced* sidebar state from Python is via set_page_config's
# initial_sidebar_state — and it only reacts when that value visibly changes
# between reruns. So toggling requires: flip to the opposite value, rerun
# once so Streamlit notices the change, then rerun again with the value we
# actually want.
_SIDEBAR_STATE = {True: "expanded", False: "collapsed"}

if "sidebar_open" not in st.session_state:
    st.session_state.sidebar_open = st.session_state.get("token") is not None
if "_sidebar_pending_rerun" not in st.session_state:
    st.session_state._sidebar_pending_rerun = False

if st.session_state._sidebar_pending_rerun:
    st.set_page_config(page_title="MoodMentor", page_icon="😊", layout="wide",
                        initial_sidebar_state=_SIDEBAR_STATE[not st.session_state.sidebar_open])
    st.session_state._sidebar_pending_rerun = False
    st.rerun()

st.set_page_config(page_title="MoodMentor", page_icon="😊", layout="wide",
                    initial_sidebar_state=_SIDEBAR_STATE[st.session_state.sidebar_open])

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

@st.cache_resource
def setup(): init_db()
setup()

# ---------------------------------------------------------------------------
# Session state
# ---------------------------------------------------------------------------
defaults = {
    "page": "login", "token": None, "email": None, "chat_history": [],
    "avatar_b64": None, "app_view": "dashboard", "last_report": None,
    "nav_history": [],
}
for k, v in defaults.items():
    if k not in st.session_state:
        st.session_state[k] = v

# Pick up page/view from the URL (lets the browser's Back/Forward buttons work)
_qp = st.query_params
if "page" in _qp and _qp["page"] != st.session_state.page:
    st.session_state.page = _qp["page"]
if "view" in _qp and _qp["view"] != st.session_state.app_view:
    st.session_state.app_view = _qp["view"]

def _current_loc():
    return ("page", st.session_state.page) if not st.session_state.token else ("view", st.session_state.app_view)

def goto(p):
    if st.session_state.page != p:
        st.session_state.nav_history.append(_current_loc())
    st.session_state.page = p
    st.query_params["page"] = p
    st.rerun()

def goto_view(v):
    if st.session_state.app_view != v:
        st.session_state.nav_history.append(_current_loc())
    st.session_state.app_view = v
    st.query_params["view"] = v
    st.rerun()

def go_back():
    if not st.session_state.nav_history:
        return
    kind, val = st.session_state.nav_history.pop()
    if kind == "page":
        st.session_state.page = val
        st.query_params["page"] = val
    else:
        st.session_state.app_view = val
        st.query_params["view"] = val
    st.rerun()

def render_sidebar_toggle():
    """Our own open/close control for the sidebar. We don't touch Streamlit's
    native collapse arrow (its behavior/test-id keeps changing between
    versions, and CSS overrides get silently re-stomped by Streamlit's own
    internal sidebar state) — instead we flip st.session_state.sidebar_open
    and let the set_page_config double-rerun trick at the top of the file
    actually force Streamlit's real sidebar state to match."""
    with st.container(key="sidebar_toggle_wrap"):
        label = "✕ Close" if st.session_state.sidebar_open else "☰ Menu"
        if st.button(label, key="sidebar_toggle_btn"):
            st.session_state.sidebar_open = not st.session_state.sidebar_open
            st.session_state._sidebar_pending_rerun = True
            st.rerun()

def render_back_button():
    """Streamlit writes to the URL with history.replaceState (not pushState),
    so the browser's real Back/Forward buttons can't step through in-app
    pages. We keep our own tiny history stack and expose it as a button."""
    if st.session_state.nav_history:
        with st.container(key=f"backwrap_{st.session_state.page}_{st.session_state.app_view}"):
            if st.button("← Back", key=f"back_{st.session_state.page}_{st.session_state.app_view}_{len(st.session_state.nav_history)}"):
                go_back()

def valid_pw(pw):
    return len(pw) >= 8 and re.search(r"[A-Za-z]", pw) and re.search(r"[0-9]", pw)

def img_to_b64(uploaded_file):
    return base64.b64encode(uploaded_file.getvalue()).decode()

def emoji_strip(emojis):
    spans = "".join(
        f'<span class="mm-emoji-chip" style="animation-delay:{i*0.25}s;">{e}</span>'
        for i, e in enumerate(emojis)
    )
    st.markdown(f'<div class="mm-emoji-strip">{spans}</div>', unsafe_allow_html=True)

# A decorative column of mood emojis pinned to the edge of the screen for the
# logged-in experience — a quieter, more "designed" alternative to a row of
# emoji sitting inline in the page flow. Purely visual, ignores clicks.
# A field of gently drifting mood emoji scattered across the whole page —
# fixed positions (not randomized) so they don't jump around on every rerun,
# weighted toward the edges/corners so they never sit on top of text.
EMOJI_FIELD_ITEMS = [
    # emoji, top%, left%, size, delay, duration, opacity
    ("😊", "6%",  "4%",  "2.6rem", "0s",    "7s",   0.9),
    ("🥰", "13%", "91%", "2.3rem", "1s",    "9s",   0.85),
    ("😌", "29%", "2%",  "2.1rem", "2s",    "8s",   0.8),
    ("😲", "43%", "95%", "2.5rem", "0.5s",  "6.5s", 0.85),
    ("😢", "59%", "3%",  "2.2rem", "1.5s",  "7.5s", 0.8),
    ("😠", "68%", "92%", "2.4rem", "2.5s",  "8.5s", 0.8),
    ("✨", "8%",  "46%", "1.8rem", "0.8s",  "6s",   0.75),
    ("💛", "85%", "9%",  "2.1rem", "1.2s",  "7s",   0.85),
    ("🌈", "88%", "80%", "2.2rem", "1.8s",  "9s",   0.8),
    ("🎉", "4%",  "68%", "2rem",   "0.3s",  "8s",   0.8),
    ("🥳", "20%", "18%", "1.9rem", "0.6s",  "7.8s", 0.75),
    ("💫", "35%", "74%", "1.9rem", "1.1s",  "6.8s", 0.75),
    ("😴", "77%", "45%", "2rem",   "1.7s",  "8.2s", 0.75),
    ("🤗", "94%", "38%", "2rem",   "2.2s",  "7.3s", 0.75),
]

def render_emoji_field():
    spans = "".join(
        f'<span style="top:{top}; left:{left}; font-size:{size}; '
        f'animation-delay:{delay}; animation-duration:{dur}; opacity:{op};">{e}</span>'
        for e, top, left, size, delay, dur, op in EMOJI_FIELD_ITEMS
    )
    st.markdown(f'<div class="mm-emoji-field">{spans}</div>', unsafe_allow_html=True)

def render_upload_ambient_bg():
    """Upload & Analyze has real content (file previews, data tables, charts)
    that the busy floating-emoji field made look cluttered, so this page gets
    something calmer instead: two large, soft, blurred color blobs sitting
    behind the content — the kind of ambient background real analytics/SaaS
    dashboards use. Purely decorative, no animation, stays out of the way."""
    st.markdown('<div class="mm-upload-bg"></div>', unsafe_allow_html=True)

INITIAL_AVATAR = (
    "data:image/svg+xml;base64," + base64.b64encode("""
    <svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 100 100'>
      <defs><linearGradient id='g' x1='0' y1='0' x2='1' y2='1'>
        <stop offset='0%' stop-color='#FFC93C'/><stop offset='100%' stop-color='#FF6F59'/>
      </linearGradient></defs>
      <rect width='100' height='100' rx='50' fill='url(#g)'/>
      <text x='50' y='66' font-size='46' text-anchor='middle'>🙂</text>
    </svg>
    """.encode("utf-8")).decode()
)

def avatar_src():
    if st.session_state.avatar_b64:
        return f"data:image/png;base64,{st.session_state.avatar_b64}"
    return INITIAL_AVATAR

# ---------------------------------------------------------------------------
# Themed plotly bar chart helper (keeps charts on-brand instead of Streamlit's
# default dark canvas)
# ---------------------------------------------------------------------------
BAR_PALETTE = ["#FFC93C", "#FF6F59", "#4EC5F1", "#4CD9A0", "#B98CE0", "#FF9AD5"]

def themed_bar(data: dict, height: int = 260):
    labels = list(data.keys())
    values = list(data.values())
    colors = [BAR_PALETTE[i % len(BAR_PALETTE)] for i in range(len(labels))]
    fig = go.Figure(go.Bar(
        x=labels, y=values, marker_color=colors,
        marker_line_width=0, text=[f"{v:.2f}" if isinstance(v, float) else v for v in values],
        textposition="outside",
    ))
    fig.update_layout(
        height=height,
        margin=dict(l=10, r=10, t=10, b=10),
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(family="Plus Jakarta Sans, sans-serif", color="#2B2140", size=13),
        yaxis=dict(showgrid=True, gridcolor="rgba(43,33,64,0.08)", zeroline=False),
        xaxis=dict(showgrid=False),
        bargap=0.35,
    )
    st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})

# ---------------------------------------------------------------------------
# Global styling
# ---------------------------------------------------------------------------
def inject_css():
    st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Fraunces:ital,wght@0,400;0,600;1,500&family=Plus+Jakarta+Sans:wght@400;500;600;700&family=JetBrains+Mono:wght@400;600&display=swap');

    :root{
        --bg-a:#FFF7E6;
        --bg-b:#EAF6FF;
        --panel: rgba(255,255,255,0.82);
        --accent-sun:#FFC93C;
        --accent-coral:#FF6F59;
        --accent-sky:#4EC5F1;
        --accent-mint:#2FBE8F;
        --ink:#2B2140;
        --ink-muted:#6B5F82;
        --border-soft: rgba(43,33,64,0.10);
    }

    html{ font-size: 19px; } /* bumps every rem-based size ~19% bigger so nothing needs manual browser zoom */
    html, body, [class*="css"]{ font-family:'Plus Jakarta Sans', sans-serif; }

    .stApp{
        background: linear-gradient(135deg, var(--bg-a) 0%, var(--bg-b) 100%);
        color: var(--ink);
    }

    #MainMenu, footer {visibility:hidden;}
    header[data-testid="stHeader"]{
        background: transparent !important;
        box-shadow: none !important;
    }
    div[data-testid="stToolbar"]{ visibility: hidden; }
    div[data-testid="stDecoration"]{ display: none; }
    /* We don't rely on Streamlit's own collapse/reopen arrow at all — its
       test-id and behavior has changed across versions and kept breaking.
       Hide it and use our own toggle button (see render_sidebar_toggle)
       which we fully control. */
    div[data-testid="collapsedControl"]{ display: none !important; }

    /* ---------- our own sidebar toggle button ---------- */
    .st-key-sidebar_toggle_wrap{
        position: fixed; top: 14px; left: 14px; z-index: 1000000;
    }
    .st-key-sidebar_toggle_wrap .stButton>button{
        background: var(--ink) !important; color:#fff !important;
        border-radius: 999px !important; font-weight:700 !important;
        padding: 0.4rem 0.9rem !important; box-shadow: 0 6px 16px rgba(43,33,64,0.28) !important;
    }
    .st-key-sidebar_toggle_wrap .stButton>button *{ color:#fff !important; }
    .st-key-sidebar_toggle_wrap .stButton>button:hover{
        transform: translateY(-1px) !important;
    }

    /* Fluid, correctly-scaled container — fills the viewport without needing
       browser zoom, but stays readable on large monitors */
    html, body{ height:100%; }
    .stApp{ min-height:100vh; }

    .block-container{
        max-width: min(750px, 90vw);
        margin: 0 auto;
        padding-top: 2rem !important;
        padding-bottom: 3rem !important;
    }

    h1,h2,h3, .mm-display{
        font-family:'Fraunces', serif !important;
        font-weight:600 !important;
        letter-spacing:-0.01em;
        color: var(--ink) !important;
    }
    h1{ font-size: clamp(1.6rem, 2.4vw, 2.3rem) !important; line-height:1.25 !important; }
    h2{ font-size: clamp(1.25rem, 1.8vw, 1.6rem) !important; }
    h3{ font-size: clamp(1.05rem, 1.4vw, 1.25rem) !important; }
    p, li, span, label, .stMarkdown { font-size: 1rem; color: var(--ink); }
    /* A button's label text sits in a nested <p>/<span>, and the bare rule
       above (which only targets p/span generically) was overriding any
       custom button text color back to ink — even on dark or coral custom
       buttons, making labels invisible. Force button labels to always
       inherit the button's own explicit color instead. */
    .stButton>button *{ color: inherit !important; }

    @media (max-width: 640px){
        .block-container{ max-width: 100vw; padding-left:1rem !important; padding-right:1rem !important; }
    }

    /* ---------- signature: a smiling sun (mood/happiness motif) ---------- */
    .mm-sun-wrap{ position:relative; width:100%; height:210px; display:flex; align-items:center; justify-content:center; }
    .mm-sun-rays{
        position:absolute; width:170px; height:170px;
        background: repeating-conic-gradient(var(--accent-sun) 0deg 8deg, transparent 8deg 20deg);
        border-radius:50%; opacity:0.55; filter: blur(0.5px);
        animation: mm-spin 22s linear infinite;
    }
    .mm-sun-face{
        position:relative; width:110px; height:110px; border-radius:50%;
        background: radial-gradient(circle at 35% 30%, #FFE18A 0%, var(--accent-sun) 55%, var(--accent-coral) 100%);
        box-shadow: 0 10px 30px rgba(255,111,89,0.35), inset 0 0 18px rgba(255,255,255,0.35);
        display:flex; align-items:center; justify-content:center; font-size:2.6rem;
        animation: mm-bob 3.4s ease-in-out infinite;
    }
    .mm-emoji-float{
        position:absolute; font-size:1.5rem; opacity:0.9; animation: mm-bob 3.6s ease-in-out infinite;
    }
    .mm-emoji-strip{ display:flex; gap:0.5rem; margin:0.15rem 0 0.7rem 0; }
    .mm-emoji-chip{ font-size:1.4rem; display:inline-block; animation: mm-bob 3s ease-in-out infinite; }
    @keyframes mm-spin{ from{ transform: rotate(0deg);} to{ transform: rotate(360deg);} }
    @keyframes mm-bob{ 0%,100%{ transform: translateY(0);} 50%{ transform: translateY(-8px);} }
    @keyframes mm-bob-slow{ 0%,100%{ transform: translateY(0) rotate(0deg);} 50%{ transform: translateY(-10px) rotate(6deg);} }

    /* ---------- full-page 3D floating emoji field (every page) ---------- */
    .mm-emoji-field{
        position: fixed; inset: 0; z-index: 1; pointer-events: none;
        overflow: hidden; perspective: 900px;
    }
    .mm-emoji-field span{
        position: absolute; display: inline-block; transform-style: preserve-3d;
        font-family: "Apple Color Emoji", "Segoe UI Emoji", "Noto Color Emoji",
                     "Segoe UI Symbol", "Twemoji Mozilla", sans-serif;
        filter: drop-shadow(0 4px 10px rgba(255,201,60,0.35));
        animation-name: mm-float-3d; animation-iteration-count: infinite;
        animation-timing-function: ease-in-out;
    }
    @keyframes mm-float-3d{
        0%   { transform: translateY(0)     rotate(0deg)  rotateY(0deg)   scale(1); }
        25%  { transform: translateY(-16px) rotate(10deg) rotateY(25deg)  scale(1.06); }
        50%  { transform: translateY(8px)   rotate(-8deg) rotateY(-20deg) scale(0.95); }
        75%  { transform: translateY(-10px) rotate(6deg)  rotateY(15deg)  scale(1.03); }
        100% { transform: translateY(0)     rotate(0deg)  rotateY(0deg)   scale(1); }
    }
    @media (max-width: 700px){ .mm-emoji-field{ display:none; } }

    /* ---------- calm ambient background for Upload & Analyze ---------- */
    .mm-upload-bg{
        position: fixed; inset: 0; z-index: 0; pointer-events: none; overflow: hidden;
    }
    .mm-upload-bg::before, .mm-upload-bg::after{
        content: ""; position: absolute; border-radius: 50%; filter: blur(70px);
    }
    .mm-upload-bg::before{
        width: 460px; height: 460px; top: -140px; right: -120px; opacity: 0.30;
        background: radial-gradient(circle, var(--accent-sky), transparent 70%);
    }
    .mm-upload-bg::after{
        width: 420px; height: 420px; bottom: -120px; left: -110px; opacity: 0.28;
        background: radial-gradient(circle, var(--accent-sun), transparent 70%);
    }

    /* ---------- back button ---------- */
    .st-key-backwrap_login .stButton>button, .st-key-backwrap_signup .stButton>button,
    div[class*="st-key-backwrap_"] .stButton>button{
        background: rgba(43,33,64,0.06) !important;
        color: var(--ink) !important; box-shadow:none !important;
        font-weight:600 !important; padding:0.35rem 0.9rem !important;
        border-radius:999px !important; font-size:0.88rem !important;
        margin-bottom: 0.6rem;
    }
    div[class*="st-key-backwrap_"] .stButton>button:hover{
        background: rgba(43,33,64,0.12) !important; transform: translateX(-2px) !important;
        box-shadow:none !important;
    }

    /* ---------- sidebar nav section labels ---------- */
    .mm-side-label{
        color:#9A8CBF; font-size:0.7rem; font-weight:700; letter-spacing:0.14em;
        text-transform:uppercase; margin: 0.9rem 0 0.15rem 0.35rem;
    }
    /* Settings and Log out both get the brand yellow so they read as the
       two "account-level" actions, distinct from ordinary nav items. */
    .st-key-navidle_settings .stButton>button, .st-key-navactive_settings .stButton>button{
        background: rgba(255,201,60,0.20) !important; color:#8A6A00 !important;
        border:1px solid rgba(255,201,60,0.55) !important; font-weight:700 !important;
        border-left: 3px solid var(--accent-sun) !important;
    }
    .st-key-navidle_settings .stButton>button:hover, .st-key-navactive_settings .stButton>button:hover{
        background: rgba(255,201,60,0.32) !important;
    }
    .st-key-logoutwrap .stButton>button{
        background: linear-gradient(135deg, var(--accent-sun), #FFB020) !important;
        color:#2B2140 !important;
        border:none !important; font-weight:700 !important;
        box-shadow: 0 6px 16px rgba(255,201,60,0.35) !important;
    }
    .st-key-logoutwrap .stButton>button:hover{
        transform: translateY(-1px) !important;
        box-shadow: 0 10px 22px rgba(255,201,60,0.45) !important;
    }

    /* ---------- glass card ---------- */
    .mm-card{
        background: var(--panel);
        border: 1px solid var(--border-soft);
        border-radius: 20px;
        padding: 1.6rem 1.8rem;
        backdrop-filter: blur(10px);
        box-shadow: 0 16px 40px rgba(43,33,64,0.12);
    }

    .mm-eyebrow{
        text-transform:uppercase; letter-spacing:0.16em; font-size:0.72rem;
        color: var(--accent-coral); font-weight:700; margin-bottom:0.3rem;
    }
    .mm-tagline{ color: var(--ink-muted); font-size:0.98rem; line-height:1.55; }

    /* ---------- avatar ---------- */
    .mm-avatar-ring{
        width:84px; height:84px; border-radius:50%;
        padding:4px;
        background: linear-gradient(135deg, var(--accent-sun), var(--accent-coral));
        display:flex; align-items:center; justify-content:center;
        margin: 0 auto 0.6rem auto;
        box-shadow: 0 8px 20px rgba(255,111,89,0.28);
        transition: transform .25s ease;
    }
    .mm-avatar-ring:hover{ transform: scale(1.05) rotate(-2deg); }
    .mm-avatar-ring img{ width:100%; height:100%; border-radius:50%; object-fit:cover; border:3px solid #fff; }

    /* ---------- buttons (covers both plain st.button AND form-submit
       buttons, which Streamlit renders with a different wrapper class —
       this is why login/signup buttons were showing up black before) ---------- */
    .stButton>button, div[data-testid="stForm"] button, .stFormSubmitButton>button{
        background: linear-gradient(135deg, var(--accent-sun), var(--accent-coral)) !important;
        color:#2B2140 !important; font-weight:700 !important; border:none !important; border-radius:12px !important;
        padding:0.55rem 1.2rem; transition: transform .15s ease, box-shadow .15s ease;
        box-shadow: 0 6px 16px rgba(255,111,89,0.28);
    }
    .stButton>button:hover, div[data-testid="stForm"] button:hover, .stFormSubmitButton>button:hover{
        transform: translateY(-2px) scale(1.015);
        box-shadow: 0 10px 24px rgba(255,111,89,0.4);
        color:#2B2140 !important;
    }
    .stButton>button:active{ transform: translateY(0px) scale(0.99); }

    /* nav pill buttons in sidebar */
    section[data-testid="stSidebar"] .stButton>button{
        background: transparent; color: var(--ink); text-align:left;
        box-shadow:none; font-weight:600; border-radius:10px;
        border:1px solid transparent; border-left: 3px solid transparent;
    }
    section[data-testid="stSidebar"] .stButton>button:hover{
        background: rgba(43,33,64,0.06); color: var(--ink);
        transform:none; border:1px solid var(--border-soft);
        border-left: 3px solid var(--accent-coral);
    }
    section[data-testid="stSidebar"] div[class*="st-key-navactive_"] .stButton>button{
        background: rgba(43,33,64,0.07) !important; color: var(--ink) !important;
        border-left: 3px solid var(--accent-coral) !important;
        border-top:1px solid var(--border-soft); border-bottom:1px solid var(--border-soft);
        border-right:1px solid var(--border-soft);
    }

    /* ---------- inputs ---------- */
    .stTextInput>div>div>input, .stTextArea textarea{
        background: rgba(255,255,255,0.9) !important;
        border: 1px solid var(--border-soft) !important;
        border-radius: 10px !important;
        color: var(--ink) !important;
    }
    .stTextInput>div>div>input:focus{ border-color: var(--accent-coral) !important; }

    /* ---------- chat input (was invisible: dark theme default made the
       typed text and its floating bottom bar both render near-black) ---------- */
    div[data-testid="stBottom"], div[data-testid="stBottomBlockContainer"],
    .stChatFloatingInputContainer, div[data-testid="stChatInput"]{
        background: var(--bg-a) !important;
    }
    div[data-testid="stChatInput"]{
        border-top: 1px solid var(--border-soft) !important;
        box-shadow: none !important;
    }
    div[data-testid="stChatInput"] > div{
        background: #fff !important;
        border: 1px solid var(--border-soft) !important;
        border-radius: 14px !important;
    }
    div[data-testid="stChatInput"] textarea{
        color: var(--ink) !important;
        -webkit-text-fill-color: var(--ink) !important;
        background: transparent !important;
        caret-color: var(--ink) !important;
    }
    div[data-testid="stChatInput"] textarea::placeholder{
        color: var(--ink-muted) !important; opacity: 0.7 !important;
    }

    /* ---------- sidebar ---------- */
    section[data-testid="stSidebar"]{
        background: linear-gradient(180deg, #FFFDF8 0%, #FBF3E4 100%);
        border-right: 1px solid var(--border-soft);
    }
    section[data-testid="stSidebar"] .mm-tagline{ color: var(--ink-muted); }

    /* ---------- stat / dashboard cards with 3D tilt ---------- */
    .mm-stat{
        background: var(--panel); border:1px solid var(--border-soft);
        border-radius:16px; padding:1.1rem 1.3rem; transition: transform .25s ease, box-shadow .25s ease;
        transform-style: preserve-3d;
    }
    .mm-stat:hover{
        transform: perspective(600px) rotateX(3deg) rotateY(-3deg) translateY(-4px);
        box-shadow: 0 18px 36px rgba(43,33,64,0.18);
        border-color: var(--accent-sun);
    }
    .mm-stat-num{ font-family:'JetBrains Mono', monospace; font-size:1.5rem; font-weight:700;
        color: var(--accent-coral); }
    .mm-stat-label{ color: var(--ink-muted); font-size:0.8rem; text-transform:uppercase; letter-spacing:0.06em;}

    /* ---------- chat bubbles ---------- */
    .mm-bubble-user{
        background: linear-gradient(135deg, var(--accent-sun), var(--accent-coral));
        color:#2B2140; padding:0.65rem 1rem; border-radius:16px 16px 4px 16px;
        max-width:80%; margin-left:auto; font-weight:500;
    }
    .mm-bubble-bot{
        background: #fff; border:1px solid var(--border-soft);
        color: var(--ink); padding:0.65rem 1rem; border-radius:16px 16px 16px 4px;
        max-width:80%;
    }

    .mm-badge{
        display:inline-block; padding:0.18rem 0.65rem; border-radius:999px;
        font-size:0.72rem; font-weight:700; letter-spacing:0.03em;
        background: rgba(47,190,143,0.14); color: #1C8F6B; border:1px solid rgba(47,190,143,0.35);
    }

    hr.mm-hr{ border:none; border-top:1px solid var(--border-soft); margin:1.1rem 0; }

    /* ---------- plain text / code output (st.text, st.write on strings) ----------
       Streamlit's default theme colors these light, which was invisible against
       our light background. Force them dark and give pre/code a visible chip.
       IMPORTANT: st.text() renders its content inside a nested <pre>/<div>
       that Streamlit's own built-in stylesheet colors explicitly (inherited
       from the light/dark base theme), with higher CSS specificity than a
       bare "pre, code" rule — so every descendant needs to be targeted
       directly, not just the wrapping container. This is why only emoji
       (which use color-glyph fonts and ignore the CSS `color` property)
       were visible before, while the surrounding words were invisible. */
    div[data-testid="stText"], div[data-testid="stMarkdownContainer"] p,
    div[data-testid="stMarkdownContainer"] li, div[data-testid="stCaptionContainer"],
    .stMarkdown, .stText, .stCaption{
        color: var(--ink) !important;
    }
    div[data-testid="stText"] pre,
    div[data-testid="stText"] *{
        color: var(--ink) !important;
        background: transparent !important;
        white-space: pre-wrap !important;
    }
    pre, code{
        color: var(--ink) !important;
        background: rgba(43,33,64,0.05) !important;
        border-radius: 8px !important;
    }
    div[data-testid="stFileUploaderFileName"]{ color: var(--ink) !important; }

    /* Belt-and-braces fallback: anything inside the main block that doesn't
       set its own color explicitly (buttons, sidebar, bubbles all have
       higher-specificity rules above and win over this) falls back to ink
       instead of whatever the base Streamlit theme would otherwise pick. */
    .main .block-container{ color: var(--ink); }
    </style>
    """, unsafe_allow_html=True)

inject_css()
# NOTE: the full-page floating 3D emoji field used to render behind every
# logged-in page. Turned off for a cleaner, more professional look now that
# there's real content (dashboards, forms, charts) on those pages. The sun
# animation stays on the login screen as the brand's signature mood motif.
# Call render_emoji_field() again here if you'd like the field back.

# ---------------------------------------------------------------------------
# LOGGED-IN EXPERIENCE
# ---------------------------------------------------------------------------
if st.session_state.token:
    user = read_token(st.session_state.token)
    if user:
        render_sidebar_toggle()
        with st.sidebar:
            st.markdown(f"""
            <div style="text-align:center; padding:0.8rem 0 0.2rem 0;">
                <div style="font-family:'Fraunces',serif; font-weight:600; font-size:1.3rem; color:var(--ink); letter-spacing:-0.01em;">
                    😊 MoodMentor
                </div>
            </div>
            <div style="text-align:center; padding:0.6rem 0 0.4rem 0;">
                <div class="mm-avatar-ring"><img src="{avatar_src()}"></div>
                <div style="font-family:'Fraunces',serif; font-size:1.05rem; color:var(--ink);">{user['username']}</div>
                <div style="color:var(--ink-muted); font-size:0.78rem;">{user['email']}</div>
            </div>
            <hr class="mm-hr">
            """, unsafe_allow_html=True)

            nav_sections = [
                ("MAIN", [("dashboard", "🏠", "Home")]),
                ("TOOLS", [("upload", "📤", "Upload & Analyze"), ("chat", "💬", "Wellness Chat")]),
                ("ACCOUNT", [("profile", "👤", "Profile"), ("settings", "⚙️", "Settings")]),
                ("SUPPORT", [("feedback", "⭐", "Feedback"), ("help", "❓", "Help & Support")]),
            ]
            for section_label, items in nav_sections:
                st.markdown(f'<div class="mm-side-label">{section_label}</div>', unsafe_allow_html=True)
                for key, icon, label in items:
                    is_active = st.session_state.app_view == key
                    wrap_key = f"navactive_{key}" if is_active else f"navidle_{key}"
                    with st.container(key=wrap_key):
                        if st.button(f"{icon}  {label}", key=f"nav_{key}", use_container_width=True):
                            goto_view(key)

            st.markdown("<hr class='mm-hr'>", unsafe_allow_html=True)
            with st.container(key="logoutwrap"):
                if st.button("🚪  Log out", key="nav_logout", use_container_width=True):
                    st.session_state.token = None
                    st.session_state.avatar_b64 = None
                    st.session_state.nav_history = []
                    goto("login")
            st.markdown('<div style="text-align:center; color:var(--ink-muted); font-size:0.72rem; '
                        'margin-top:0.8rem;">MoodMentor · made with 💛</div>', unsafe_allow_html=True)

        view = st.session_state.app_view
        render_back_button()

        # ---------------- DASHBOARD ----------------
        if view == "dashboard":
            st.markdown(f"""
            <div class="mm-eyebrow">Welcome back</div>
            <h1 style="margin-top:0;">Hi {user['username']}! 👋😊</h1>
            <p class="mm-tagline">Here's a quick look at your wellness space today.</p>
            """, unsafe_allow_html=True)

            c1, c2, c3 = st.columns(3)
            with c1:
                st.markdown(f"""<div class="mm-stat">
                    <div class="mm-stat-label">💬 Chat Messages</div>
                    <div class="mm-stat-num">{len(st.session_state.chat_history)}</div>
                </div>""", unsafe_allow_html=True)
            with c2:
                last = st.session_state.last_report
                sentiment = last["final_sentiment"] if last else "—"
                st.markdown(f"""<div class="mm-stat">
                    <div class="mm-stat-label">🙂 Last Sentiment</div>
                    <div class="mm-stat-num" style="font-size:1.3rem;">{sentiment}</div>
                </div>""", unsafe_allow_html=True)
            with c3:
                emotion = last["final_emotion"] if last else "—"
                st.markdown(f"""<div class="mm-stat">
                    <div class="mm-stat-label">💛 Dominant Emotion</div>
                    <div class="mm-stat-num" style="font-size:1.3rem;">{emotion}</div>
                </div>""", unsafe_allow_html=True)

            st.markdown("<hr class='mm-hr'>", unsafe_allow_html=True)
            qc1, qc2 = st.columns(2)
            with qc1:
                st.markdown("""<div class="mm-card">
                    <div class="mm-eyebrow">Quick action</div>
                    <h3 style="margin-top:0;">📤 Analyze new feedback</h3>
                    <p class="mm-tagline">Upload a CSV or TXT file and run the NLP pipeline.</p>
                </div>""", unsafe_allow_html=True)
                if st.button("Go to Upload & Analyze", key="qa_upload"):
                    goto_view("upload")
            with qc2:
                st.markdown("""<div class="mm-card">
                    <div class="mm-eyebrow">Quick action</div>
                    <h3 style="margin-top:0;">💬 Talk it through</h3>
                    <p class="mm-tagline">Open a supportive chat about how you're feeling.</p>
                </div>""", unsafe_allow_html=True)
                if st.button("Go to Wellness Chat", key="qa_chat"):
                    goto_view("chat")

        # ---------------- PROFILE ----------------
        elif view == "profile":
            st.markdown('<div class="mm-eyebrow">Account</div><h1 style="margin-top:0;">Your profile 👤</h1>',
                        unsafe_allow_html=True)
            pc1, pc2 = st.columns([1, 2])
            with pc1:
                st.markdown(f'<div class="mm-avatar-ring"><img src="{avatar_src()}"></div>',
                            unsafe_allow_html=True)
                new_pic = st.file_uploader("Change profile picture", type=["png", "jpg", "jpeg"],
                                            key="profile_pic_uploader")
                if new_pic is not None:
                    st.session_state.avatar_b64 = img_to_b64(new_pic)
                    save_avatar(user["email"], st.session_state.avatar_b64)
                    st.success("Profile picture updated.")
                    st.rerun()
                st.caption("Photos are saved to your account and stay across logins.")
            with pc2:
                st.markdown(f"""<div class="mm-card">
                    <p class="mm-tagline"><b>Username</b></p><p>{user['username']}</p>
                    <p class="mm-tagline"><b>Email</b></p><p>{user['email']}</p>
                    <span class="mm-badge">VERIFIED ✓</span>
                </div>""", unsafe_allow_html=True)

            st.markdown("<hr class='mm-hr'>", unsafe_allow_html=True)
            st.markdown("""<div class="mm-card">
                <h3 style="margin-top:0;">🔒 Change password</h3>
                <p class="mm-tagline">Enter your current password, then choose a new one.</p>
            </div>""", unsafe_allow_html=True)
            with st.form("change_pw_form"):
                cur_pw = st.text_input("Current password", type="password")
                new_pw1 = st.text_input("New password", type="password",
                                         placeholder="8+ chars, letters & numbers")
                new_pw2 = st.text_input("Confirm new password", type="password")
                pw_go = st.form_submit_button("Update password →")
            if pw_go:
                full_user = get_user(user["email"])
                if not full_user or not check_pw(cur_pw, full_user["password_hash"]):
                    st.error("Current password is incorrect.")
                elif not valid_pw(new_pw1):
                    st.error("New password needs 8+ chars, letters and numbers.")
                elif new_pw1 != new_pw2:
                    st.error("New passwords don't match.")
                else:
                    set_password(user["email"], new_pw1)
                    st.success("Password updated.")

        # ---------------- CHAT ----------------
        elif view == "chat":
            st.markdown('<div class="mm-eyebrow">A space to talk</div>'
                        '<h1 style="margin-top:0;">💬 Wellness Chat</h1>'
                        '<p class="mm-tagline">A supportive space to talk about how you\'re feeling. '
                        'Not a substitute for professional care.</p>', unsafe_allow_html=True)

            chat_box = st.container(height=420)
            with chat_box:
                for turn in st.session_state.chat_history:
                    if turn["role"] == "user":
                        st.markdown(f'<div style="display:flex; margin:0.4rem 0;">'
                                    f'<div class="mm-bubble-user">{turn["content"]}</div></div>',
                                    unsafe_allow_html=True)
                    else:
                        st.markdown(f'<div style="display:flex; margin:0.4rem 0;">'
                                    f'<div class="mm-bubble-bot">🙂 {turn["content"]}</div></div>',
                                    unsafe_allow_html=True)

            user_msg = st.chat_input("How are you feeling today?")
            if user_msg:
                st.session_state.chat_history.append({"role": "user", "content": user_msg})
                recent_history = st.session_state.chat_history[-10:-1]
                try:
                    resp = requests.post(
                        f"{BACKEND_URL}/chat",
                        json={"message": user_msg, "history": recent_history},
                        headers={"Authorization": f"Bearer {st.session_state.token}"},
                        timeout=60,
                    )
                    reply = resp.json()["reply"] if resp.status_code == 200 else \
                        "Sorry, I couldn't reach the wellness assistant right now."
                except requests.exceptions.RequestException:
                    reply = "Sorry, I couldn't reach the wellness assistant right now."
                st.session_state.chat_history.append({"role": "assistant", "content": reply})
                st.rerun()

            if st.session_state.chat_history and st.button("Clear chat"):
                st.session_state.chat_history = []
                st.rerun()

        # ---------------- UPLOAD & ANALYZE ----------------
        elif view == "upload":
            render_upload_ambient_bg()
            st.markdown('<div class="mm-eyebrow">NLP pipeline</div>'
                        '<h1 style="margin-top:0;">📤 Upload & Analyze</h1>',
                        unsafe_allow_html=True)

            uploaded = st.file_uploader("Choose a CSV or TXT file", type=["csv", "txt"])
            headers = {"Authorization": f"Bearer {st.session_state.token}"}

            if uploaded is not None:
                is_csv = uploaded.name.lower().endswith(".csv")
                column_name = None
                if is_csv:
                    column_name = st.text_input(
                        "Feedback column name (leave blank to use the last column)"
                    ).strip() or None

                c1, c2 = st.columns(2)

                if c1.button("Upload & Preview"):
                    files = {"file": (uploaded.name, uploaded.getvalue())}
                    try:
                        resp = requests.post(f"{BACKEND_URL}/upload", files=files,
                                              headers=headers, timeout=15)
                    except requests.exceptions.RequestException as e:
                        st.error(f"Could not reach backend: {e}")
                        resp = None

                    if resp is not None:
                        if resp.status_code == 200:
                            data = resp.json()
                            st.success(f"Uploaded '{data['filename']}' — {data['row_count']} data row(s). 🎉")
                            if data["type"] == "csv" and data["columns"]:
                                st.markdown(f"**Columns:** {', '.join(data['columns'])}")
                                if data["preview_rows"]:
                                    st.dataframe(
                                        [dict(zip(data["columns"], row)) for row in data["preview_rows"]]
                                    )
                            elif data["preview_lines"]:
                                st.markdown(
                                    '<div class="mm-card" style="padding:1rem 1.2rem;">' +
                                    "<br>".join(str(line) for line in data["preview_lines"]) +
                                    "</div>",
                                    unsafe_allow_html=True,
                                )
                        else:
                            try:
                                st.error(resp.json().get("detail", "Upload failed."))
                            except ValueError:
                                st.error(f"Upload failed (status {resp.status_code}).")

                if c2.button("Run NLP Analysis"):
                    files = {"file": (uploaded.name, uploaded.getvalue())}
                    form = {"column": column_name} if column_name else {}
                    with st.spinner("Running multilingual NLP pipeline… this can take a moment. ✨"):
                        try:
                            resp = requests.post(f"{BACKEND_URL}/analyze", files=files, data=form,
                                                  headers=headers, timeout=120)
                        except requests.exceptions.RequestException as e:
                            st.error(f"Could not reach backend: {e}")
                            resp = None

                    if resp is not None:
                        if resp.status_code != 200:
                            try:
                                st.error(resp.json().get("detail", "Analysis failed."))
                            except ValueError:
                                st.error(f"Analysis failed (status {resp.status_code}).")
                        else:
                            r = resp.json()
                            st.session_state.last_report = r
                            st.markdown('<hr class="mm-hr">', unsafe_allow_html=True)
                            st.markdown('<h2>📋 Employee Wellness Analysis Report</h2>', unsafe_allow_html=True)

                            m1, m2 = st.columns(2)
                            m1.markdown(f"**File:** {r['filename']}  \n**Type:** {r['file_type']}")
                            if r.get("used_column"):
                                m2.markdown(f"**Column used:** {r['used_column']}")

                            st.markdown(f"**🌐 Detected language:** {r['detected_language']} "
                                        f"({r['language_code']})")

                            with st.expander("Cleaned text"):
                                st.write(r["cleaned_text"])
                            with st.expander("Sentences"):
                                for i, s in enumerate(r["sentences"], 1):
                                    st.write(f"{i}. {s}")
                            with st.expander("Tokens (original vs. filtered)"):
                                st.markdown(f"**Original:** {r['original_tokens']}")
                                st.markdown(f"**Filtered:** {r['filtered_tokens']}")
                            with st.expander("Translation & lemmatization"):
                                st.markdown(f"**English translation:** {r['translated_text']}")
                                st.markdown(f"**Lemmatized:** {r['lemmatized_text']}")

                            st.markdown(f"**Emojis found:** "
                                        f"{', '.join(r['emoji_list']) if r['emoji_list'] else 'None'}")

                            scores = r["sentiment_scores"]
                            st.markdown(f'<span class="mm-badge">SENTIMENT · {r["final_sentiment"].upper()}</span>',
                                        unsafe_allow_html=True)
                            themed_bar({
                                "Positive 😊": scores["pos"],
                                "Negative 😔": scores["neg"],
                                "Neutral 😐": scores["neu"],
                            })
                            st.caption(f"Compound score: {scores['compound']:.3f}")

                            st.markdown(f'<span class="mm-badge">EMOTION · {r["final_emotion"].upper()}</span>',
                                        unsafe_allow_html=True)
                            themed_bar(r["emotion_scores"])

                            s1, s2, s3, s4 = st.columns(4)
                            for col, label, val in zip(
                                (s1, s2, s3, s4),
                                ("Characters", "Sentences", "Orig. tokens", "Filtered tokens"),
                                (r["original_char_count"], len(r["sentences"]),
                                 len(r["original_tokens"]), len(r["filtered_tokens"])),
                            ):
                                col.markdown(f"""<div class="mm-stat">
                                    <div class="mm-stat-label">{label}</div>
                                    <div class="mm-stat-num">{val}</div>
                                </div>""", unsafe_allow_html=True)

        # ---------------- SETTINGS ----------------
        elif view == "settings":
            st.markdown('<div class="mm-eyebrow">Preferences</div><h1 style="margin-top:0;">⚙️ Settings</h1>',
                        unsafe_allow_html=True)

            st.markdown("""<div class="mm-card">
                <h3 style="margin-top:0;">🔔 Notifications</h3>
                <p class="mm-tagline">Choose what MoodMentor keeps you posted on.</p>
            </div>""", unsafe_allow_html=True)
            st.session_state.setdefault("pref_email_notifs", True)
            st.session_state.setdefault("pref_weekly_summary", True)
            st.session_state.pref_email_notifs = st.toggle(
                "Email me when a new analysis report is ready",
                value=st.session_state.pref_email_notifs)
            st.session_state.pref_weekly_summary = st.toggle(
                "Send a weekly mood summary",
                value=st.session_state.pref_weekly_summary)

            st.markdown("<hr class='mm-hr'>", unsafe_allow_html=True)
            st.markdown("""<div class="mm-card">
                <h3 style="margin-top:0;">🔒 Account & security</h3>
                <p class="mm-tagline">Password resets are handled from the login screen for now —
                log out and choose "Forgot password?" to change yours.</p>
            </div>""", unsafe_allow_html=True)

            st.markdown("<hr class='mm-hr'>", unsafe_allow_html=True)
            with st.expander("⚠️ Danger zone"):
                st.markdown('<p class="mm-tagline">Deleting your account removes your chat '
                            'history and reports. This can\'t be undone.</p>', unsafe_allow_html=True)
                st.button("Request account deletion", key="delete_account_btn")

        # ---------------- FEEDBACK ----------------
        elif view == "feedback":
            st.markdown('<div class="mm-eyebrow">Tell us what you think</div>'
                        '<h1 style="margin-top:0;">⭐ Feedback</h1>'
                        '<p class="mm-tagline">Your ratings and comments help us improve MoodMentor.</p>',
                        unsafe_allow_html=True)

            st.session_state.setdefault("fb_rating", 5)
            st.markdown('<div class="mm-card">', unsafe_allow_html=True)
            st.markdown("**How would you rate MoodMentor?**")
            star_cols = st.columns(5)
            for i, col in enumerate(star_cols, start=1):
                filled = i <= st.session_state.fb_rating
                if col.button("⭐" if filled else "☆", key=f"star_{i}", use_container_width=True):
                    st.session_state.fb_rating = i
                    st.rerun()
            st.caption(f"{st.session_state.fb_rating} / 5")

            with st.form("feedback_form"):
                msg = st.text_area("What's working well, or what should we fix/add?",
                                    placeholder="e.g. Loved the dashboard, but I'd like dark mode…")
                fb_go = st.form_submit_button("Submit feedback →")
            if fb_go:
                if not msg.strip():
                    st.error("Please add a short message before submitting.")
                else:
                    save_feedback(user["email"], user["username"], st.session_state.fb_rating, msg.strip())
                    st.success("Thanks! Your feedback has been recorded. 💛")
                    st.session_state.fb_rating = 5
            st.markdown('</div>', unsafe_allow_html=True)

        # ---------------- HELP & SUPPORT ----------------
        elif view == "help":
            st.markdown('<div class="mm-eyebrow">We\'re here</div><h1 style="margin-top:0;">❓ Help & Support</h1>',
                        unsafe_allow_html=True)

            faqs = [
                ("What does MoodMentor do?",
                 "It reads journal entries or feedback text and reports the sentiment "
                 "and emotion behind it, so you can spot patterns over time."),
                ("Is my chat private?",
                 "Your chat and reports are tied to your account and only visible to you "
                 "when you're logged in."),
                ("Can I use it in another language?",
                 "Yes — the Upload & Analyze pipeline detects the language automatically "
                 "and translates it before scoring."),
            ]
            for q, a in faqs:
                with st.expander(q):
                    st.markdown(f'<p class="mm-tagline">{a}</p>', unsafe_allow_html=True)

            st.markdown("<hr class='mm-hr'>", unsafe_allow_html=True)
            st.markdown("""<div class="mm-card">
                <h3 style="margin-top:0;">📮 Still stuck?</h3>
                <p class="mm-tagline">Reach out any time and we'll get back to you.</p>
            </div>""", unsafe_allow_html=True)
            st.link_button("Email support", "mailto:support@moodmentor.app")

            st.markdown("<hr class='mm-hr'>", unsafe_allow_html=True)
            st.caption("MoodMentor is a supportive space, not a substitute for professional "
                       "care. If you're going through something difficult, please also reach "
                       "out to a licensed professional or a crisis line in your area.")
        st.stop()
    st.session_state.token = None

# ---------------------------------------------------------------------------
# LOGGED-OUT EXPERIENCE (login / signup / verify / forgot / reset)
# ---------------------------------------------------------------------------
render_back_button()

left, right = st.columns([1, 1.15], gap="large")

with left:
    st.markdown("""
    <div class="mm-sun-wrap">
        <div class="mm-sun-rays"></div>
        <div class="mm-sun-face">🙂</div>
        <div class="mm-emoji-float" style="top:8%; left:12%; animation-delay:0.3s;">😌</div>
        <div class="mm-emoji-float" style="top:15%; right:10%; animation-delay:1.1s;">😲</div>
        <div class="mm-emoji-float" style="bottom:6%; left:20%; animation-delay:0.6s;">😢</div>
        <div class="mm-emoji-float" style="bottom:10%; right:16%; animation-delay:1.6s;">🥰</div>
    </div>
    <div class="mm-eyebrow" style="text-align:center;">MoodMentor</div>
    <h1 style="text-align:center; margin-top:0;">Understand your <i>mood</i>,<br>one word at a time 😊</h1>
    <p class="mm-tagline" style="text-align:center;">AI-powered emotional tone analysis and
    personalized well-being recommendations — for journals, feedback, and how you're feeling right now.</p>
    """, unsafe_allow_html=True)

with right:
    st.markdown('<div class="mm-card">', unsafe_allow_html=True)

    if st.session_state.page == "login":
        st.markdown('<div class="mm-eyebrow">Welcome back</div><h2 style="margin-top:0;">Log in 👋</h2>',
                    unsafe_allow_html=True)
        with st.form("login"):
            email = st.text_input("Email", placeholder="you@example.com")
            pw = st.text_input("Password", type="password", placeholder="••••••••")
            go = st.form_submit_button("Log in →", use_container_width=True)
        if go:
            user = get_user(email.strip().lower())
            if not user or not check_pw(pw, user["password_hash"]):
                st.error("Invalid email or password.")
            elif not user["is_verified"]:
                st.warning("Verify your email first.")
                st.session_state.email = user["email"]; goto("verify")
            else:
                st.session_state.token = make_token(user)
                st.session_state.app_view = "dashboard"
                st.session_state.nav_history = []
                st.session_state.avatar_b64 = load_avatar(user["email"])
                st.rerun()

        c1, c2 = st.columns(2)
        if c1.button("Create account", use_container_width=True): goto("signup")
        if c2.button("Forgot password?", use_container_width=True): goto("forgot")

    elif st.session_state.page == "signup":
        emoji_strip(["✨", "📝", "🙂"])
        st.markdown('<div class="mm-eyebrow">New here</div><h2 style="margin-top:0;">Create your account ✨</h2>',
                    unsafe_allow_html=True)

        st.markdown(f'<div class="mm-avatar-ring"><img src="{avatar_src()}"></div>', unsafe_allow_html=True)
        pic = st.file_uploader("Add a profile picture (optional)", type=["png", "jpg", "jpeg"],
                                key="signup_pic")
        if pic is not None:
            st.session_state.avatar_b64 = img_to_b64(pic)
            st.rerun()

        with st.form("signup"):
            username = st.text_input("Username", placeholder="e.g. dishi_codes")
            email = st.text_input("Email", placeholder="you@example.com")
            pw = st.text_input("Password", type="password", placeholder="8+ chars, letters & numbers")
            go = st.form_submit_button("Create account →", use_container_width=True)
        if go:
            email = email.strip().lower()
            if len(username) < 3:
                st.error("Username too short.")
            elif not valid_pw(pw):
                st.error("Password needs 8+ chars, letters and numbers.")
            elif username_taken(username) or get_user(email):
                st.error("Username or email already in use.")
            else:
                create_user(username, email, pw)
                code = new_otp(); save_otp(email, code, "signup")
                ok, msg = send_otp(email, code, "signup")
                if ok:
                    st.session_state.email = email
                    st.success("Check your email for the code. 📬")
                    goto("verify")
                else:
                    st.error(f"Email failed: {msg}")
        if st.button("← Back to login"): goto("login")

    elif st.session_state.page == "verify":
        emoji_strip(["✅", "📬", "🔐"])
        st.markdown('<div class="mm-eyebrow">Almost there</div><h2 style="margin-top:0;">Verify your email ✅</h2>',
                    unsafe_allow_html=True)
        email = st.session_state.email
        st.markdown(f'<p class="mm-tagline">Enter the code sent to <b>{email}</b></p>', unsafe_allow_html=True)
        with st.form("verify"):
            code = st.text_input("6-digit code", max_chars=6, placeholder="••••••")
            go = st.form_submit_button("Verify →", use_container_width=True)
        if go:
            if check_otp(email, code.strip(), "signup"):
                verify_user(email)
                st.success("Verified! Please log in.")
                goto("login")
            else:
                st.error("Invalid or expired code.")
        if st.button("← Back"): goto("login")

    elif st.session_state.page == "forgot":
        emoji_strip(["🔑", "📮", "🤝"])
        st.markdown('<div class="mm-eyebrow">No worries</div><h2 style="margin-top:0;">Reset your password 🔑</h2>',
                    unsafe_allow_html=True)
        with st.form("forgot"):
            email = st.text_input("Your account email", placeholder="you@example.com")
            go = st.form_submit_button("Send reset code →", use_container_width=True)
        if go:
            email = email.strip().lower()
            if get_user(email):
                code = new_otp(); save_otp(email, code, "password_reset")
                send_otp(email, code, "password_reset")
            st.session_state.email = email
            st.info("If that email exists, a code was sent.")
            goto("reset")
        if st.button("← Back"): goto("login")

    elif st.session_state.page == "reset":
        emoji_strip(["🔒", "🌱", "🙌"])
        st.markdown('<div class="mm-eyebrow">Last step</div><h2 style="margin-top:0;">Set a new password 🔒</h2>',
                    unsafe_allow_html=True)
        email = st.session_state.email
        with st.form("reset"):
            code = st.text_input("Reset code", max_chars=6, placeholder="••••••")
            pw = st.text_input("New password", type="password", placeholder="8+ chars, letters & numbers")
            go = st.form_submit_button("Reset password →", use_container_width=True)
        if go:
            if not valid_pw(pw):
                st.error("Password needs 8+ chars, letters and numbers.")
            elif not check_otp(email, code.strip(), "password_reset"):
                st.error("Invalid or expired code.")
            else:
                set_password(email, pw)
                st.success("Password reset. Please log in.")
                goto("login")
        if st.button("← Back"): goto("login")

    st.markdown('</div>', unsafe_allow_html=True)


Writing app.py


## FastAPI backend (JWT-protected file upload)

This backend exposes a `/upload` endpoint that only accepts `.csv` or `.txt` files, verifies the same JWT issued at Streamlit login, and returns a preview (columns + first rows for CSV, first lines for TXT).


## Multilingual NLP pipeline module

Language detection, text cleaning, Telugu/Kannada stopword filtering, translation to English, lemmatization, VADER sentiment, and keyword-based emotion detection — imported by `backend.py`'s `/analyze` endpoint.


In [8]:
%%writefile nlp_pipeline.py
"""
nlp_pipeline.py
Multilingual NLP pipeline for employee feedback:
normalize -> detect language -> clean -> tokenize -> stopword-filter ->
translate to English -> lemmatize -> sentiment (VADER) -> emotion (BERT).

Stopword filtering uses the `stopwordsiso` package, which ships stopword
sets for 50+ languages keyed by ISO 639-1 code (the same codes langdetect
returns), so any supported language is handled automatically instead of
needing a hardcoded list per language. If the detected language isn't in
stopwordsiso's coverage, filtering is simply skipped for that text.

Heavy libs (spacy model, translator, vader, BERT emotion model, Qwen chat
model) load once at import time via lazy module-level globals, so repeated
/analyze calls reuse them.
"""

import re
import ftfy
import emoji
import spacy
import torch
import stopwordsiso
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline as hf_pipeline,
)
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

DetectorFactory.seed = 0

_nlp = None
_vader = None
_qwen_model = None
_qwen_tokenizer = None
_bert_emotion_pipeline = None

QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Fine-tuned BERT model for emotion classification, trained on the
# GoEmotions dataset (28 fine-grained emotion labels). This directly
# matches the project brief's requirement for a "fine-tuned BERT model"
# for multi-label emotion classification.
BERT_EMOTION_MODEL_NAME = "bhadresh-savani/bert-base-go-emotion"

# Human-readable names for common languages this pipeline is likely to see.
# Purely cosmetic (shown in the UI) -- falls back to the raw ISO code if a
# language isn't listed here, so it never blocks processing of any language.
LANGUAGE_NAMES = {
    "te": "Telugu", "kn": "Kannada", "en": "English", "ta": "Tamil",
    "hi": "Hindi", "ml": "Malayalam", "mr": "Marathi", "bn": "Bengali", "gu": "Gujarati",
    "fr": "French", "de": "German", "es": "Spanish", "pt": "Portuguese",
    "ar": "Arabic", "zh": "Chinese", "ja": "Japanese", "ko": "Korean", "ru": "Russian",
}


def _get_stopwords(language_code: str) -> set:
    """
    Returns the stopword set for `language_code` using stopwordsiso, which
    covers 50+ languages by ISO 639-1 code. Returns an empty set for any
    language it doesn't cover -- filtering is skipped rather than failing,
    so unsupported languages still flow through the rest of the pipeline.
    """
    if stopwordsiso.has_lang(language_code):
        return stopwordsiso.stopwords(language_code)
    return set()

# Fixed label set the app's UI (charts, backend.py) is built around. Keeping
# this list stable means downstream code never has to change, no matter
# which underlying model produces the emotion.
EMOTION_LABELS = ["Happy", "Sad", "Stress", "Angry", "Fear", "Neutral"]

EMOTION_EMOJI = {
    "Happy": "\U0001F60A", "Sad": "\U0001F622", "Stress": "\U0001F62B",
    "Angry": "\U0001F621", "Fear": "\U0001F628", "Neutral": "\U0001F610",
}

# GoEmotions ships 28 fine-grained labels. We map each one to the closest
# of our 6 app-level labels so the rest of the app (charts, DB columns,
# Streamlit UI) never has to know GoEmotions exists. Anything not listed
# here (e.g. "curiosity", "confusion") falls back to "Neutral".
GOEMOTIONS_TO_APP_LABEL = {
    "joy": "Happy", "amusement": "Happy", "excitement": "Happy",
    "love": "Happy", "gratitude": "Happy", "optimism": "Happy",
    "relief": "Happy", "pride": "Happy", "admiration": "Happy",
    "approval": "Happy", "caring": "Happy",

    "sadness": "Sad", "disappointment": "Sad", "grief": "Sad",
    "remorse": "Sad",

    "nervousness": "Stress", "embarrassment": "Stress",
    "confusion": "Stress",

    "anger": "Angry", "annoyance": "Angry", "disgust": "Angry",
    "disapproval": "Angry",

    "fear": "Fear",

    "neutral": "Neutral", "realization": "Neutral", "surprise": "Neutral",
    "curiosity": "Neutral", "desire": "Neutral",
}


def _get_nlp():
    """Lazy-load the multilingual spaCy model once per process."""
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("xx_sent_ud_sm")
    return _nlp


def _get_vader():
    global _vader
    if _vader is None:
        _vader = SentimentIntensityAnalyzer()
    return _vader


def _get_qwen():
    """Lazy-load Qwen2.5-0.5B-Instruct once per process (GPU if available).
    Still used by the wellness chatbot (wellness_chat_reply) -- only the
    emotion-detection step now uses BERT instead."""
    global _qwen_model, _qwen_tokenizer
    if _qwen_model is None:
        _qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
        _qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
    return _qwen_model, _qwen_tokenizer


def _get_bert_emotion_pipeline():
    """
    Lazy-load the fine-tuned BERT emotion classifier once per process, using
    Hugging Face's `pipeline()` helper -- this bundles the tokenizer and the
    model together so we just call it with raw text and get scores back.

    `top_k=None` tells the pipeline to return a score for every label
    instead of just the single top prediction, so we can build a full
    scores dict (matching what the UI already expects).
    """
    global _bert_emotion_pipeline
    if _bert_emotion_pipeline is None:
        _bert_emotion_pipeline = hf_pipeline(
            "text-classification",
            model=BERT_EMOTION_MODEL_NAME,
            top_k=None,  # return all label scores, not just the best one
            device=0 if torch.cuda.is_available() else -1,
        )
    return _bert_emotion_pipeline


def _bert_emotion(text: str) -> dict:
    """
    Classifies `text` using the fine-tuned BERT GoEmotions model, then maps
    the 28 GoEmotions labels down to our 6 app-level EMOTION_LABELS by
    summing mapped scores. Returns the same shape the rest of the app
    already expects: {"emotion": <label>, "scores": {label: 0-1, ...}}.
    """
    classifier = _get_bert_emotion_pipeline()

    if not text.strip():
        text = "(empty feedback)"

    # top_k=None returns a list of {"label": ..., "score": ...} dicts
    # covering all 28 GoEmotions labels, e.g.:
    # [{"label": "joy", "score": 0.82}, {"label": "neutral", "score": 0.05}, ...]
    raw_predictions = classifier(text, truncation=True)[0]

    app_scores = {label: 0.0 for label in EMOTION_LABELS}
    for pred in raw_predictions:
        goemotion_label = pred["label"].lower()
        app_label = GOEMOTIONS_TO_APP_LABEL.get(goemotion_label, "Neutral")
        app_scores[app_label] += pred["score"]

    # Normalize so scores are readable as relative proportions (0-1 range).
    total = sum(app_scores.values()) or 1.0
    app_scores = {label: round(score / total, 4) for label, score in app_scores.items()}

    final_emotion = max(app_scores, key=app_scores.get)
    return {"emotion": final_emotion, "scores": app_scores}


def process_employee_feedback(text: str) -> dict:
    """Runs the full pipeline on a single blob of text and returns a results dict."""
    nlp = _get_nlp()
    vader = _get_vader()

    normalized_text = ftfy.fix_text(text)

    try:
        language = detect(normalized_text)
    except Exception:
        language = "unknown"
    detected_language = LANGUAGE_NAMES.get(language, "Other / Unknown")

    emoji_list = [ch for ch in normalized_text if ch in emoji.EMOJI_DATA]

    cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", normalized_text)
    cleaned_text = re.sub(r"\S+@\S+", " ", cleaned_text)
    cleaned_text = re.sub(r"@\w+|#\w+", " ", cleaned_text)
    cleaned_text = emoji.replace_emoji(cleaned_text, replace="")
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    doc = nlp(cleaned_text)
    sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
    original_tokens = [t.text for t in doc if not t.is_space]
    clean_tokens = [t.text for t in doc if not t.is_punct and not t.is_space and not t.like_num]

    selected_stopwords = _get_stopwords(language)
    filtered_tokens = [t for t in clean_tokens if t.lower() not in selected_stopwords]
    final_preprocessed_text = " ".join(filtered_tokens)

    try:
        translated_text = GoogleTranslator(source="auto", target="en").translate(final_preprocessed_text)
    except Exception as error:
        translated_text = f"Translation failed: {error}"

    english_doc = nlp(translated_text)
    lemmas = [t.lemma_ if t.lemma_ else t.text for t in english_doc if not t.is_space]
    lemmatized_text = " ".join(lemmas)

    sentiment_scores = vader.polarity_scores(translated_text)
    compound_score = sentiment_scores["compound"]
    if compound_score >= 0.05:
        final_sentiment = "Positive \U0001F60A"
    elif compound_score <= -0.05:
        final_sentiment = "Negative \U0001F614"
    else:
        final_sentiment = "Neutral \U0001F610"

    # --- Emotion detection via fine-tuned BERT (GoEmotions) ---
    # Replaces the earlier Qwen-prompting approach: BERT classifies
    # directly against fixed labels instead of generating free-form text
    # that then has to be parsed as JSON, so it's faster and more reliable.
    bert_result = _bert_emotion(translated_text)
    emotion_scores = bert_result["scores"]
    final_emotion_label = bert_result["emotion"]
    final_emotion = f"{final_emotion_label} {EMOTION_EMOJI.get(final_emotion_label, '')}"

    return {
        "language_code": language,
        "detected_language": detected_language,
        "normalized_text": normalized_text,
        "cleaned_text": cleaned_text,
        "sentences": sentences,
        "original_tokens": original_tokens,
        "filtered_tokens": filtered_tokens,
        "emoji_list": emoji_list,
        "final_preprocessed_text": final_preprocessed_text,
        "translated_text": translated_text,
        "lemmatized_text": lemmatized_text,
        "sentiment_scores": sentiment_scores,
        "final_sentiment": final_sentiment,
        "emotion_scores": emotion_scores,
        "final_emotion": final_emotion,
    }


# --- Crisis-keyword safety net for the wellness chatbot -----------------
# This is a simple, deliberately blunt keyword check that runs regardless
# of what the LLM says. If it fires, we always show crisis resources —
# we never rely on the small LLM alone to catch something this important.
CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "want to die", "self harm",
    "self-harm", "hurt myself", "not worth living", "no reason to live",
]

CRISIS_MESSAGE = (
    "I'm really glad you reached out, and I want to make sure you get support "
    "beyond what I can offer here. If you're in immediate danger, please contact "
    "your local emergency number right now. You can also reach a crisis line: "
    "in India, AASRA is available at +91-9820466726 (24/7). If you're outside "
    "India, please look up a local crisis helpline or talk to a trusted person "
    "or your HR/EAP contact. You don't have to go through this alone."
)

WELLNESS_SYSTEM_PROMPT = (
    "You are a supportive workplace wellness assistant for employees. "
    "Your role is to listen, validate feelings, and offer general, gentle "
    "coping suggestions (like breathing exercises, taking a short break, "
    "or talking to a trusted colleague or manager). "
    "You are NOT a therapist or doctor: never diagnose any condition, never "
    "claim expertise you don't have, and never give medical or medication "
    "advice. If the employee describes something serious (ongoing crisis, "
    "self-harm, harming others), gently encourage them to contact a mental "
    "health professional, their HR/EAP program, or a crisis helpline. "
    "Keep replies short (2-4 sentences), warm, and non-judgmental. "
    "Avoid clinical labels and avoid being preachy or repetitive."
)


def _contains_crisis_language(text: str) -> bool:
    lowered = text.lower()
    return any(kw in lowered for kw in CRISIS_KEYWORDS)


def wellness_chat_reply(message: str, history: list[dict] | None = None) -> dict:
    """
    Generates a supportive wellness chatbot reply using the Qwen chat model.
    (The chatbot still uses Qwen -- it needs to generate free-form
    conversational replies, which is a generation task, not a
    classification task, so BERT isn't a fit here.)

    `history` is an optional list of {"role": "user"|"assistant", "content": str}
    dicts representing prior turns in the conversation (kept short/recent by
    the caller — this function does not trim it).

    Always checks for crisis language first; if found, returns a fixed,
    resource-pointing message instead of an LLM-generated one, since we
    never want a small model improvising in a safety-critical moment.
    """
    if _contains_crisis_language(message):
        return {"reply": CRISIS_MESSAGE, "flagged": True}

    model, tokenizer = _get_qwen()

    messages = [{"role": "system", "content": WELLNESS_SYSTEM_PROMPT}]
    for turn in (history or []):
        if turn.get("role") in ("user", "assistant") and turn.get("content"):
            messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(generated, skip_special_tokens=True).strip()

    if not reply:
        reply = "I'm here and listening — could you tell me a bit more about how you're feeling?"

    return {"reply": reply, "flagged": False}


Writing nlp_pipeline.py


In [9]:
%%writefile backend.py
import os, io, jwt, csv
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
from nlp_pipeline import process_employee_feedback, wellness_chat_reply
load_dotenv()

SECRET = os.getenv("JWT_SECRET")
app = FastAPI(title="Upload API")

app.add_middleware(CORSMiddleware, allow_origins=["*"],
                    allow_methods=["*"], allow_headers=["*"])

def get_user(authorization: str = Header(None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing token")
    token = authorization.split(" ", 1)[1]
    try:
        return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(401, "Invalid or expired token")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/upload")
async def upload(file: UploadFile = File(...), authorization: str = Header(None)):
    user = get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024  # 5 MB cap
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    lines = text.splitlines()
    row_count = len(lines)
    preview_lines = lines[:20]

    columns = None
    preview_rows = None
    if ext == "csv":
        reader = csv.reader(io.StringIO(text))
        rows = list(reader)
        if rows:
            columns = rows[0]
            preview_rows = rows[1:21]
            row_count = max(len(rows) - 1, 0)  # exclude header from data row count

    return {
        "filename": name,
        "type": ext,
        "uploaded_by": user["username"],
        "row_count": row_count,
        "columns": columns,
        "preview_rows": preview_rows,
        "preview_lines": None if ext == "csv" else preview_lines,
    }


def _extract_text_blob(raw: bytes, ext: str, column: str | None) -> tuple[str, str | None]:
    """
    Returns (text_blob, used_column). For TXT, used_column is None.
    For CSV, joins all non-empty values of the chosen column (or the last
    column if none/invalid was specified) into one whitespace-joined blob —
    matches the notebook's "whole file as one blob" behavior.
    """
    text = raw.decode("utf-8")

    if ext == "txt":
        return text.strip(), None

    reader = csv.reader(io.StringIO(text))
    rows = list(reader)
    if not rows:
        raise HTTPException(400, "CSV file has no rows.")

    header = rows[0]
    data_rows = rows[1:]
    if not data_rows:
        raise HTTPException(400, "CSV file has a header but no data rows.")

    col_index = None
    if column and column in header:
        col_index = header.index(column)
    else:
        col_index = len(header) - 1  # default to last column

    values = [row[col_index] for row in data_rows if len(row) > col_index and row[col_index].strip()]
    blob = " ".join(values).strip()
    if not blob:
        raise HTTPException(400, f"Column '{header[col_index]}' has no readable text.")
    return blob, header[col_index]


@app.post("/analyze")
async def analyze(file: UploadFile = File(...), column: str = Form(None),
                   authorization: str = Header(None)):
    """
    Runs the multilingual NLP pipeline (language detection, cleaning,
    stopword filtering, translation, lemmatization, VADER sentiment,
    keyword-based emotion) on an uploaded .csv or .txt file.
    """
    get_user(authorization)  # just verifies the token; raises 401 if invalid

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text_blob, used_column = _extract_text_blob(raw, ext, column)
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    results = process_employee_feedback(text_blob)
    results["filename"] = name
    results["file_type"] = ext.upper()
    results["used_column"] = used_column
    results["original_char_count"] = len(text_blob)
    return results


class ChatTurn(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    message: str
    history: list[ChatTurn] = []


@app.post("/chat")
async def chat(payload: ChatRequest, authorization: str = Header(None)):
    """
    Wellness support chatbot endpoint. Stateless on the server: the client
    (Streamlit) sends the recent conversation history along with each new
    message, and we generate the next reply with the same Qwen model used
    for emotion detection.
    """
    get_user(authorization)  # verifies the token; raises 401 if invalid

    message = payload.message.strip()
    if not message:
        raise HTTPException(400, "Message cannot be empty.")

    history = [turn.dict() for turn in payload.history]
    result = wellness_chat_reply(message, history=history)
    return result


Writing backend.py


In [10]:
from db import init_db
init_db()
print("✅ Connected to PostgreSQL and ensured tables exist.")

✅ Connected to PostgreSQL and ensured tables exist.


In [11]:
from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = values["NGROK_AUTHTOKEN"]

# Kill any previous tunnels/streamlit/uvicorn instances from earlier runs in this session
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
time.sleep(1)

# Launch FastAPI (backend.py) in the background on port 8000 (internal only, not tunneled)
get_ipython().system_raw(
    'uvicorn backend:app --host 0.0.0.0 --port 8000 &'
)
time.sleep(5)  # NLP libs (spaCy model etc.) take a little longer to import

# Launch Streamlit in the background, quietly, on port 8501
get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 --server.headless true '
    '--server.enableCORS false --server.enableXsrfProtection false &'
)
time.sleep(4)  # give both servers a moment to boot

public_url = ngrok.connect(8501, "http")
print(f"🚀 Your app is live at: {public_url}")
print("FastAPI backend is running internally on port 8000 (Streamlit talks to it via localhost).")
print("Open the URL above in your browser. Leave this Colab cell/runtime running to keep it up.")

🚀 Your app is live at: NgrokTunnel: "https://unclaimed-fragment-cherisher.ngrok-free.dev" -> "http://localhost:8501"
FastAPI backend is running internally on port 8000 (Streamlit talks to it via localhost).
Open the URL above in your browser. Leave this Colab cell/runtime running to keep it up.


In [ ]:
from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
print("Stopped Streamlit, FastAPI, and closed ngrok tunnel.")

Stopped Streamlit, FastAPI, and closed ngrok tunnel.
